# Conteo de palabras con Hadoop

# 1. Activar servicios Hadoop (SSH, HDFS, YARN)

Ejecuta cada celda para activar los servicios necesarios antes de cargar datos a HDFS.


## 1.1 Iniciar el servicio SSH

In [10]:
import subprocess
import getpass
try:
    print("Iniciando el servicio SSH...")
    password = getpass.getpass("Introduce la contraseña de sudo: ")
    result = subprocess.run(['sudo', '-S', 'service', 'ssh', 'start'], input=password+'\n', encoding='utf-8', check=True, capture_output=True)
    print(result.stdout)
    print("Servicio SSH iniciado.")
except subprocess.CalledProcessError as e:
    print(f"Error al iniciar SSH: {e}")
    if e.stderr:
        print(e.stderr)

Iniciando el servicio SSH...
 * Starting OpenBSD Secure Shell server sshd
   ...done.

Servicio SSH iniciado.


## 1.2 Probar conexión SSH local

In [11]:
import subprocess
try:
    print("Probando conexión SSH local...")
    subprocess.run(['ssh', '-o', 'StrictHostKeyChecking=no', 'localhost', 'exit'], check=True)
    print("Conexión SSH local exitosa.")
except subprocess.CalledProcessError as e:
    print(f"Error en conexión SSH local: {e}")


Probando conexión SSH local...
Conexión SSH local exitosa.


## 1.3 Detener e iniciar HDFS

In [12]:
import subprocess
import os
import shutil
try:
    print("Deteniendo HDFS (por si está corriendo)...")
    subprocess.run(['stop-dfs.sh'], check=True)
    # Limpieza automática del datanode si hay incompatibilidad de clusterID
    datanode_dir = '/home/esilvas/hdfs/datanode'
    if os.path.exists(datanode_dir):
        # Buscar si existe VERSION con clusterID
        version_file = os.path.join(datanode_dir, 'current', 'VERSION')
        if os.path.exists(version_file):
            with open(version_file, 'r', encoding='utf-8', errors='replace') as f:
                content = f.read()
            if 'clusterID' in content:
                print("Eliminando datos antiguos del DataNode para evitar incompatibilidad de clusterID...")
                shutil.rmtree(datanode_dir)
                os.makedirs(datanode_dir, exist_ok=True)
                print("Directorio datanode limpiado.")
    print("Iniciando HDFS (NameNode, DataNode, SecondaryNameNode)...")
    subprocess.run(['start-dfs.sh'], check=True)
    print("HDFS iniciado.")
except subprocess.CalledProcessError as e:
    print(f"Error al iniciar HDFS: {e}")
except Exception as e:
    print(f"Error en la limpieza automática del datanode: {e}")

Deteniendo HDFS (por si está corriendo)...
Stopping namenodes on [ESILVAS02]
Stopping datanodes
Stopping secondary namenodes [ESILVAS02]
Eliminando datos antiguos del DataNode para evitar incompatibilidad de clusterID...
Directorio datanode limpiado.
Iniciando HDFS (NameNode, DataNode, SecondaryNameNode)...
Starting namenodes on [ESILVAS02]
Starting datanodes
Starting secondary namenodes [ESILVAS02]
HDFS iniciado.


## 1.4 Detener e iniciar YARN

In [13]:

try:
    print("Deteniendo YARN (por si está corriendo)...")
    subprocess.run(['stop-yarn.sh'], check=True)
    print("Iniciando YARN (ResourceManager, NodeManager)...")
    subprocess.run(['start-yarn.sh'], check=True)
    print("YARN iniciado.")
except subprocess.CalledProcessError as e:
    print(f"Error al iniciar YARN: {e}")


Deteniendo YARN (por si está corriendo)...
Stopping nodemanagers
Stopping resourcemanager
Iniciando YARN (ResourceManager, NodeManager)...
Starting resourcemanager
Starting nodemanagers
YARN iniciado.


# 2. Comprobar servicios Hadoop


## 2.1 Verificar procesos de Hadoop con jps

In [26]:

try:
    print("Verificando procesos Java activos (Hadoop)...")
    subprocess.run(['jps'], check=True)
    print("Procesos Java verificados.")
except subprocess.CalledProcessError as e:
    print(f"Error al verificar procesos Java: {e}")

Verificando procesos Java activos (Hadoop)...
21936 Jps
16177 NodeManager
14899 NameNode
15924 ResourceManager
15095 DataNode
15341 SecondaryNameNode
Procesos Java verificados.


## 2.2 Verificar versión de Hadoop

In [15]:

try:
    print("\nVerificando versión de Hadoop...")
    subprocess.run(['hadoop', 'version'], check=True)
    print("Versión de Hadoop verificada.")
except subprocess.CalledProcessError as e:
    print(f"Error al verificar versión de Hadoop: {e}")




Verificando versión de Hadoop...
Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /usr/local/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar
Versión de Hadoop verificada.


## 2.3 Listar directorio raíz de HDFS

In [16]:

try:
    print("\nListando directorio raíz de HDFS...")
    subprocess.run(['hdfs', 'dfs', '-ls', '/'], check=True)
    print("Directorio raíz de HDFS listado.")
except subprocess.CalledProcessError as e:
    print(f"Error al listar directorio raíz de HDFS: {e}")




Listando directorio raíz de HDFS...
Directorio raíz de HDFS listado.


## 2.4 Verificar el reporte de HDFS

In [17]:

try:
    print("\nReporte del sistema de archivos HDFS...")
    subprocess.run(['hdfs', 'dfsadmin', '-report'], check=True)
    print("Reporte de HDFS generado.")
except subprocess.CalledProcessError as e:
    print(f"Error al generar reporte de HDFS: {e}")




Reporte del sistema de archivos HDFS...
Configured Capacity: 269490393088 (250.98 GB)
Present Capacity: 248129150976 (231.09 GB)
DFS Remaining: 248129126400 (231.09 GB)
DFS Used: 24576 (24 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost)
Hostname: ESILVAS02.cens.corp.epm.com.co
Decommission Status : Normal
Configured Capacity: 269490393088 (250.98 GB)
DFS Used: 24576 (24 KB)
Non DFS Used: 7600570368 (7.08 GB)
DFS Remaining: 248129126400 (231.09 GB)
DFS Used%: 0.00%


## 2.5 Puertos NameNode y ResourceManager

In [18]:
print("🟢 HDFS NameNode: http://localhost:9870")
print("🟢 YARN ResourceManager: http://localhost:8088")

🟢 HDFS NameNode: http://localhost:9870
🟢 YARN ResourceManager: http://localhost:8088


# 3. Convierte un archivo PDF a texto plano usando pdftotext


In [19]:
import subprocess
import os   
 
 # Rutas de archivos
PDF_FILE = "/home/esilvas/Documents/hadoop-python/files/Inteligencia_Rodriguez_ICE_2022.pdf"
OUTPUT_FILE = "/home/esilvas/Documents/hadoop-python/files/texto_plano.txt"


try:
    print(f"Convirtiendo PDF a texto plano...")
    print(f"Archivo origen: {PDF_FILE}")
    print(f"Archivo destino: {OUTPUT_FILE}")
    
    # Ejecutar pdftotext para extraer el texto
    subprocess.run([
        'pdftotext',
        PDF_FILE,
        OUTPUT_FILE
    ], check=True)
    
    # Verificar que se creó el archivo
    if os.path.exists(OUTPUT_FILE):
        # Obtener tamaño del archivo
        size = os.path.getsize(OUTPUT_FILE)
        print(f"\nConversión exitosa!")
        print(f"Archivo creado: {OUTPUT_FILE}")
        print(f"Tamaño: {size} bytes")
        
        # Contar líneas y palabras
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            words = sum(len(line.split()) for line in lines)
        
        print(f"Líneas: {len(lines)}")
        print(f"Palabras aproximadas: {words}")
    else:
        print("Error: No se pudo crear el archivo de salida")
        
except subprocess.CalledProcessError as e:
    print(f"Error al ejecutar pdftotext: {e}")
except Exception as e:
    print(f"Error: {e}")


Convirtiendo PDF a texto plano...
Archivo origen: /home/esilvas/Documents/hadoop-python/files/Inteligencia_Rodriguez_ICE_2022.pdf
Archivo destino: /home/esilvas/Documents/hadoop-python/files/texto_plano.txt

Conversión exitosa!
Archivo creado: /home/esilvas/Documents/hadoop-python/files/texto_plano.txt
Tamaño: 66525 bytes
Líneas: 1116
Palabras aproximadas: 9574


# 4. Carga de datos a HDFS

Este notebook permite cargar el archivo `texto_plano.txt` a HDFS ejecutando cada paso de forma individual.

## 4.1 Verificar que el archivo local existe

Este paso verifica que el archivo `texto_plano.txt` esté presente en la ruta local antes de cargarlo a HDFS.

In [20]:
import os

LOCAL_FILE = "/home/esilvas/Documents/hadoop-python/files/texto_plano.txt"

if os.path.exists(LOCAL_FILE):
    print(f"El archivo local existe: {LOCAL_FILE}")
else:
    print(f"El archivo local NO existe: {LOCAL_FILE}")

El archivo local existe: /home/esilvas/Documents/hadoop-python/files/texto_plano.txt


## 4.2 Crear directorio en HDFS

Este paso crea el directorio de entrada en HDFS si no existe.

In [21]:
import subprocess

HDFS_DIR = "/user/esilvas/wordcount/input"

try:
    print(f"Creando directorio en HDFS: {HDFS_DIR}")
    subprocess.run(['hdfs', 'dfs', '-mkdir', '-p', HDFS_DIR], check=True)
    print("Directorio creado o ya existente.")
except subprocess.CalledProcessError as e:
    print(f"Error al crear directorio en HDFS: {e}")

Creando directorio en HDFS: /user/esilvas/wordcount/input
Directorio creado o ya existente.


## 4.3 Subir archivo a HDFS

Este paso sube el archivo de texto plano a HDFS para que pueda ser procesado por Hadoop.

In [22]:
HDFS_FILE = f"{HDFS_DIR}/texto_plano.txt"

try:
    print(f"Subiendo archivo a HDFS: {HDFS_FILE}")
    subprocess.run(['hdfs', 'dfs', '-put', '-f', LOCAL_FILE, HDFS_FILE], check=True)
    print("Archivo subido correctamente.")
except subprocess.CalledProcessError as e:
    print(f"Error al subir archivo a HDFS: {e}")

Subiendo archivo a HDFS: /user/esilvas/wordcount/input/texto_plano.txt
Archivo subido correctamente.


## 4.4 Verificar archivo en HDFS

Este paso lista el contenido del directorio en HDFS para confirmar que el archivo fue subido correctamente.

In [23]:
try:
    print(f"Verificando archivo en HDFS: {HDFS_DIR}")
    subprocess.run(['hdfs', 'dfs', '-ls', HDFS_DIR], check=True)
except subprocess.CalledProcessError as e:
    print(f"Error al verificar archivo en HDFS: {e}")

Verificando archivo en HDFS: /user/esilvas/wordcount/input
Found 1 items
-rw-r--r--   1 esilvas supergroup      66525 2025-12-17 10:06 /user/esilvas/wordcount/input/texto_plano.txt


# 5. Mapper

El proceso Mapper es la primera fase del modelo MapReduce. Su función es leer cada línea del archivo de entrada, dividirla en palabras y emitir cada palabra junto con el valor 1. Esto representa la ocurrencia de cada palabra en el texto.

A continuación, se muestra un ejemplo de implementación de un Mapper en Python para el conteo de palabras. Este script será utilizado por Hadoop Streaming para procesar el archivo de texto cargado en HDFS.

In [ ]:
# mapper.py
import sys

for line in sys.stdin:
    # Elimina espacios en blanco al inicio y final
    line = line.strip()
    # Divide la línea en palabras
    words = line.split()
    # Emite cada palabra con el valor 1
    for word in words:
        print(f"{word}\t1")

# 6. Reducer

El Reducer es la segunda fase del modelo MapReduce. Su función es recibir las salidas del Mapper agrupadas por palabra, sumar los valores y emitir el total de ocurrencias de cada palabra.

A continuación, se muestra un ejemplo de implementación de un Reducer en Python para el conteo de palabras. Este script será utilizado por Hadoop Streaming para procesar la salida del Mapper.

In [ ]:
# reducer.py
import sys

current_word = None
current_count = 0
word = None

for line in sys.stdin:
    line = line.strip()
    word, count = line.split('\t', 1)
    try:
        count = int(count)
    except ValueError:
        continue
    if current_word == word:
        current_count += count
    else:
        if current_word:
            print(f"{current_word}\t{current_count}")
        current_word = word
        current_count = count
if current_word == word:
    print(f"{current_word}\t{current_count}")

# 7. Ejecución del trabajo MapReduce en Hadoop Streaming

En este paso, se ejecuta el trabajo MapReduce usando Hadoop Streaming, especificando los scripts de Mapper y Reducer creados anteriormente. El archivo de entrada es el texto cargado en HDFS y la salida será un archivo de texto con el conteo de palabras.

El resultado se puede descargar y visualizar como un archivo `.txt` o `.csv` para su análisis.

In [35]:
# Eliminar el directorio de salida en HDFS si existe para evitar errores en Hadoop Streaming
import subprocess

HDFS_OUTPUT = "/user/esilvas/wordcount/output"

try:
    print(f"Eliminando directorio de salida en HDFS si existe: {HDFS_OUTPUT}")
    subprocess.run(["hdfs", "dfs", "-rm", "-r", "-skipTrash", HDFS_OUTPUT], check=True)
    print("Directorio de salida eliminado.")
except subprocess.CalledProcessError as e:
    if "No such file or directory" in str(e):
        print("El directorio de salida no existía, se puede continuar.")
    else:
        print(f"Error al eliminar el directorio de salida: {e}")

Eliminando directorio de salida en HDFS si existe: /user/esilvas/wordcount/output
Deleted /user/esilvas/wordcount/output
Directorio de salida eliminado.


In [36]:
# Guardar los scripts mapper.py y reducer.py en la raíz del proyecto y verificar su existencia
import os
import stat

# Definir rutas absolutas para los scripts en la raíz del proyecto
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == 'code' else os.getcwd()
mapper_path = os.path.join(project_root, 'mapper.py')
reducer_path = os.path.join(project_root, 'reducer.py')

# Guardar mapper.py
with open(mapper_path, 'w', encoding='utf-8') as f:
    f.write('''import sys\nfor line in sys.stdin:\n    line = line.strip()\n    words = line.split()\n    for word in words:\n        print(f"{word}\\t1")\n''')

# Guardar reducer.py
with open(reducer_path, 'w', encoding='utf-8') as f:
    f.write('''import sys\ncurrent_word = None\ncurrent_count = 0\nword = None\nfor line in sys.stdin:\n    line = line.strip()\n    word, count = line.split('\\t', 1)\n    try:\n        count = int(count)\n    except ValueError:\n        continue\n    if current_word == word:\n        current_count += count\n    else:\n        if current_word:\n            print(f"{current_word}\\t{current_count}")\n        current_word = word\n        current_count = count\nif current_word == word:\n    print(f"{current_word}\\t{current_count}")\n''')

# Hacer ejecutables los scripts
os.chmod(mapper_path, os.stat(mapper_path).st_mode | stat.S_IXUSR)
os.chmod(reducer_path, os.stat(reducer_path).st_mode | stat.S_IXUSR)

# Verificar existencia y permisos
mapper_exists = os.path.exists(mapper_path) and os.access(mapper_path, os.X_OK)
reducer_exists = os.path.exists(reducer_path) and os.access(reducer_path, os.X_OK)

if mapper_exists and reducer_exists:
    print(f"Scripts guardados y son ejecutables en la raíz del proyecto:\n- {mapper_path}\n- {reducer_path}")
else:
    print(f"Error: No se pudieron guardar o hacer ejecutables los scripts.\nmapper.py existe: {os.path.exists(mapper_path)}, ejecutable: {os.access(mapper_path, os.X_OK)}\nreducer.py existe: {os.path.exists(reducer_path)}, ejecutable: {os.access(reducer_path, os.X_OK)}")

Scripts guardados y son ejecutables en la raíz del proyecto:
- /home/esilvas/Documents/hadoop-python/mapper.py
- /home/esilvas/Documents/hadoop-python/reducer.py


In [37]:
# Verificar y probar scripts mapper.py y reducer.py en la raíz del proyecto
import os
import stat
import subprocess

project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == 'code' else os.getcwd()
mapper_path = os.path.join(project_root, 'mapper.py')
reducer_path = os.path.join(project_root, 'reducer.py')

print(f"Verificando existencia y permisos de los scripts en: {project_root}")

for script in [mapper_path, reducer_path]:
    exists = os.path.exists(script)
    is_exec = os.access(script, os.X_OK)
    print(f"{os.path.basename(script)} existe: {exists}, ejecutable: {is_exec}")
    if not exists:
        print(f"❌ El archivo {script} no existe.")
    if exists and not is_exec:
        print(f"⚠️ El archivo {script} no es ejecutable. Corrigiendo permisos...")
        os.chmod(script, os.stat(script).st_mode | stat.S_IXUSR)
        print(f"Permiso de ejecución añadido a {script}.")

# Probar mapper.py
if os.path.exists(mapper_path):
    print("\nProbando mapper.py con entrada de ejemplo:")
    try:
        result = subprocess.run(['python3', mapper_path], input="hola mundo mundo\n", text=True, capture_output=True, check=True)
        print("Salida:")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error al ejecutar mapper.py: {e}\nSalida de error:\n{e.stderr}")
else:
    print("mapper.py no existe, no se puede probar.")

# Probar reducer.py
if os.path.exists(reducer_path):
    print("\nProbando reducer.py con entrada de ejemplo:")
    try:
        input_text = "hola\t1\nmundo\t1\nmundo\t1\n"
        result = subprocess.run(['python3', reducer_path], input=input_text, text=True, capture_output=True, check=True)
        print("Salida:")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error al ejecutar reducer.py: {e}\nSalida de error:\n{e.stderr}")
else:
    print("reducer.py no existe, no se puede probar.")

Verificando existencia y permisos de los scripts en: /home/esilvas/Documents/hadoop-python
mapper.py existe: True, ejecutable: True
reducer.py existe: True, ejecutable: True

Probando mapper.py con entrada de ejemplo:
Salida:
hola	1
mundo	1
mundo	1


Probando reducer.py con entrada de ejemplo:
Salida:
hola	1
mundo	2



In [38]:
# Mostrar el contenido de mapper.py y reducer.py y probarlos localmente
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == 'code' else os.getcwd()
mapper_path = os.path.join(project_root, 'mapper.py')
reducer_path = os.path.join(project_root, 'reducer.py')

print("Contenido de mapper.py:\n-------------------------")
if os.path.exists(mapper_path):
    with open(mapper_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("mapper.py no existe.")

print("\nContenido de reducer.py:\n-------------------------")
if os.path.exists(reducer_path):
    with open(reducer_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("reducer.py no existe.")

# Probar mapper.py y reducer.py como en la celda anterior
import subprocess

if os.path.exists(mapper_path):
    print("\nProbando mapper.py con entrada de ejemplo:")
    try:
        result = subprocess.run(['python3', mapper_path], input="hola mundo mundo\n", text=True, capture_output=True, check=True)
        print("Salida:")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error al ejecutar mapper.py: {e}\nSalida de error:\n{e.stderr}")
else:
    print("mapper.py no existe, no se puede probar.")

if os.path.exists(reducer_path):
    print("\nProbando reducer.py con entrada de ejemplo:")
    try:
        input_text = "hola\t1\nmundo\t1\nmundo\t1\n"
        result = subprocess.run(['python3', reducer_path], input=input_text, text=True, capture_output=True, check=True)
        print("Salida:")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"❌ Error al ejecutar reducer.py: {e}\nSalida de error:\n{e.stderr}")
else:
    print("reducer.py no existe, no se puede probar.")

Contenido de mapper.py:
-------------------------
import sys
for line in sys.stdin:
    line = line.strip()
    words = line.split()
    for word in words:
        print(f"{word}\t1")


Contenido de reducer.py:
-------------------------
import sys
current_word = None
current_count = 0
word = None
for line in sys.stdin:
    line = line.strip()
    word, count = line.split('\t', 1)
    try:
        count = int(count)
    except ValueError:
        continue
    if current_word == word:
        current_count += count
    else:
        if current_word:
            print(f"{current_word}\t{current_count}")
        current_word = word
        current_count = count
if current_word == word:
    print(f"{current_word}\t{current_count}")


Probando mapper.py con entrada de ejemplo:
Salida:
hola	1
mundo	1
mundo	1


Probando reducer.py con entrada de ejemplo:
Salida:
hola	1
mundo	2



## Verificar y subir archivo de entrada a HDFS

Estas celdas permiten:
- Listar el contenido del directorio de entrada en HDFS para confirmar si el archivo `texto_plano.txt` está presente.
- Subir el archivo desde `./files/texto_plano.txt` a la ruta correcta en HDFS si es necesario.

In [39]:
# Listar el contenido del directorio de entrada en HDFS
def listar_hdfs_input():
    import subprocess
    HDFS_DIR = "/user/esilvas/wordcount/input"
    print(f"\nListando archivos en {HDFS_DIR} (HDFS):")
    try:
        subprocess.run(["hdfs", "dfs", "-ls", HDFS_DIR], check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error al listar archivos en HDFS: {e}")

listar_hdfs_input()


Listando archivos en /user/esilvas/wordcount/input (HDFS):
Found 1 items
-rw-r--r--   1 esilvas supergroup      66525 2025-12-17 10:06 /user/esilvas/wordcount/input/texto_plano.txt


In [40]:
# Subir el archivo ./files/texto_plano.txt a la ruta correcta en HDFS si es necesario
import os
import subprocess

# Ruta local absoluta del archivo fuente
LOCAL_FILE = os.path.abspath(os.path.join(os.getcwd(), "../files/texto_plano.txt"))
HDFS_DIR = "/user/esilvas/wordcount/input"
HDFS_FILE = f"{HDFS_DIR}/texto_plano.txt"

print(f"\nVerificando archivo local: {LOCAL_FILE}")
if os.path.exists(LOCAL_FILE):
    print(f"El archivo local existe: {LOCAL_FILE}")
    print(f"Subiendo a HDFS: {HDFS_FILE}")
    try:
        subprocess.run(["hdfs", "dfs", "-put", "-f", LOCAL_FILE, HDFS_FILE], check=True)
        print("Archivo subido correctamente a HDFS.")
    except subprocess.CalledProcessError as e:
        print(f"Error al subir archivo a HDFS: {e}")
else:
    print("El archivo local NO existe. Verifica la ruta.")


Verificando archivo local: /home/esilvas/Documents/hadoop-python/files/texto_plano.txt
El archivo local existe: /home/esilvas/Documents/hadoop-python/files/texto_plano.txt
Subiendo a HDFS: /user/esilvas/wordcount/input/texto_plano.txt
Archivo subido correctamente a HDFS.


## Comando Hadoop Streaming con rutas absolutas

Puedes ejecutar el siguiente comando en la terminal para lanzar el trabajo MapReduce, asegurando que los scripts `mapper.py` y `reducer.py` se tomen desde la ruta local correcta:

```bash
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
    -input /user/esilvas/wordcount/input/texto_plano.txt \
    -output /user/esilvas/wordcount/output \
    -mapper "python3 /home/esilvas/Documents/hadoop-python/mapper.py" \
    -reducer "python3 /home/esilvas/Documents/hadoop-python/reducer.py"
```

- No es necesario subir los scripts a HDFS, solo deben estar accesibles en el sistema de archivos local donde ejecutas el comando.
- El archivo de entrada sí debe estar en HDFS.
- Si el comando termina con código 0, el trabajo fue exitoso y puedes continuar con la descarga y análisis del resultado.

## Descargar y mostrar resumen del resultado de Hadoop Streaming

Esta celda descarga el archivo de salida desde HDFS y muestra las primeras líneas para verificar el conteo de palabras.

In [41]:
# Descargar el resultado de Hadoop Streaming y mostrar un resumen
import subprocess
import os

HDFS_RESULT = "/user/esilvas/wordcount/output/part-00000"
LOCAL_RESULT = os.path.abspath("resultado_palabras.txt")

print(f"Descargando resultado desde HDFS: {HDFS_RESULT}\nA archivo local: {LOCAL_RESULT}")
try:
    subprocess.run(["hdfs", "dfs", "-get", "-f", HDFS_RESULT, LOCAL_RESULT], check=True)
    print("Descarga completada. Primeras líneas del resultado:\n")
    with open(LOCAL_RESULT, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            print(line.rstrip())
            if i >= 19:
                print("... (más líneas no mostradas)")
                break
except subprocess.CalledProcessError as e:
    print(f"Error al descargar el resultado: {e}")
except Exception as e:
    print(f"Error al leer el archivo local: {e}")

Descargando resultado desde HDFS: /user/esilvas/wordcount/output/part-00000
A archivo local: /home/esilvas/Documents/hadoop-python/code/resultado_palabras.txt
Descarga completada. Primeras líneas del resultado:

%	5
&	10
(...)».	1
(1.a	1
(20)—	1
(2010).	1
(2015).	2
(2015,	1
(2016).	4
(2017).	7
(2017,	1
(2018)	1
(2018),	1
(2018).	5
(2018,	1
(2019),	1
(2019).	2
(2019,	1
(2019a).	1
(2019a,	1
... (más líneas no mostradas)
